## Conexión a drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/Deep_learning/proyecto_final


/content/drive/MyDrive/Deep_learning/proyecto_final


## GPU

In [ ]:
import torch

print('GPU disponible:', torch.cuda.is_available())
print('Dispositivo:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

GPU disponible: True
Dispositivo: Tesla T4


## Librerias

In [ ]:
!pip install timm -q
!pip install kagglehub -q
!pip install lion-pytorch -q

## FineTuning del modelo



### Configuración del Modelo

### Arquitectura: EfficientNet-B0

Se eligió **EfficientNet-B0** por ser liviano y apto para inferencia en tiempo real,
lo cual es necesario para la demo con cámara. La diferencia de accuracy con versiones
mayores (B3, B4) no justifica el costo en velocidad para este proyecto.

### Estructura interna
```
Input [224x224x3]
       ↓
Stem (conv inicial)
       ↓
MBConv Blocks x16     ← conocimiento transferido desde ImageNet
(Mobile Inverted Bottleneck)
       ↓
Head (conv final)
       ↓
Global Average Pooling
       ↓
Dropout
       ↓
Classifier (Linear)   ← capa reemplazada
  ImageNet: 1280 → 1000 clases
  Nuestro:  1280 → 7 emociones
```

### Parámetros de `timm.create_model`

| Parámetro | Valor | Descripción |
|---|---|---|
| arquitectura | `efficientnet_b0` | Modelo base preentrenado |
| `pretrained` | `True` | Carga pesos de ImageNet |
| `num_classes` | `7` | Reemplaza la última capa automáticamente |

### Clases a predecir
`angry` · `disgust` · `fear` · `happy` · `sad` · `surprise` · `neutral`


### Estrategia de entrenamiento
Fine-tuning completo con learning rate bajo (`1e-5`) para ajustar los pesos
de ImageNet sin destruirlos, usando Lion + ReduceLROnPlateau.

In [ ]:
import timm
import torch.nn as nn

In [ ]:
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=7)
model = model.to(device)

print('Clasificador final:', model.classifier)
print('Modelo en:', next(model.parameters()).device)

Clasificador final: Linear(in_features=1280, out_features=7, bias=True)
Modelo en: cuda:0


In [ ]:
print('Clasificador final:', model.classifier)
print('Parámetros totales:    ', sum(p.numel() for p in model.parameters()))
print('Parámetros entrenables:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Clasificador final: Linear(in_features=1280, out_features=7, bias=True)
Parámetros totales:     4016515
Parámetros entrenables: 4016515


#### Importar Dataset

In [ ]:
import kagglehub
import shutil
import os

# Descargar
path = kagglehub.dataset_download("msambare/fer2013")
print("Descargado en:", path)

# Copiar a Drive
destino = '/content/drive/MyDrive/Deep_learning/proyecto_final/fer2013'
shutil.copytree(path, destino)
print("Copiado a Drive:", destino)
print(os.listdir(destino))

Using Colab cache for faster access to the 'fer2013' dataset.
Descargado en: /kaggle/input/fer2013
Copiado a Drive: /content/drive/MyDrive/Deep_learning/proyecto_final/fer2013
['test', 'train']


In [ ]:
# Verificar dataset
destino = '/content/drive/MyDrive/Deep_learning/proyecto_final/fer2013'

for split in ['train', 'test']:
    path = os.path.join(destino, split)
    print(f'\n{split}:')
    for c in sorted(os.listdir(path)):
        n = len(os.listdir(os.path.join(path, c)))
        print(f'  {c}: {n} imágenes')


train:
  angry: 3995 imágenes
  disgust: 436 imágenes
  fear: 4097 imágenes
  happy: 7215 imágenes
  neutral: 4965 imágenes
  sad: 4830 imágenes
  surprise: 3171 imágenes

test:
  angry: 958 imágenes
  disgust: 111 imágenes
  fear: 1024 imágenes
  happy: 1774 imágenes
  neutral: 1233 imágenes
  sad: 1247 imágenes
  surprise: 831 imágenes


## Análisis de Desbalance del Dataset

FER2013 presenta un desbalance significativo entre clases:

| Clase    | Train | Test |
|----------|-------|------|
| happy    | 7215  | 1774 |
| neutral  | 4965  | 1233 |
| sad      | 4830  | 1247 |
| fear     | 4097  | 1024 |
| angry    | 3995  |  958 |
| surprise | 3171  |  831 |
| disgust  |  436  |  111 |

**Problema:** disgust tiene ~16x menos imágenes que happy.
Sin corrección, el modelo aprenderá mejor happy y mal disgust.

### Estrategia de solución: Class Weights + Data Augmentation

**Class Weights:** penaliza más los errores en clases escasas dentro
del loss. A menor cantidad de imágenes → mayor peso en el loss.

**Data Augmentation:** genera variaciones artificiales de todas las
imágenes durante el entrenamiento (flip, rotación, brillo) para
aumentar la variedad visual especialmente en clases escasas.

Ambas técnicas se complementan y son el estándar en datasets desbalanceados.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

base_dir = '/content/drive/MyDrive/Deep_learning/proyecto_final/fer2013'

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(root=f'{base_dir}/train', transform=train_transforms)
test_dataset  = datasets.ImageFolder(root=f'{base_dir}/test',  transform=val_transforms)

clases = train_dataset.classes
print('Clases:', clases)
print('Total train:', len(train_dataset))
print('Total test: ', len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2)

Clases: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Total train: 28709
Total test:  7178


In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

etiquetas = [label for _, label in train_dataset.samples]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(etiquetas),
    y=etiquetas
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

print('=== Class Weights ===')
for clase, peso in zip(clases, class_weights):
    print(f'  {clase:10s}: {peso:.4f}')

=== Class Weights ===
  angry     : 1.0266
  disgust   : 9.4066
  fear      : 1.0010
  happy     : 0.5684
  neutral   : 0.8260
  sad       : 0.8491
  surprise  : 1.2934


## Análisis de Class Weights

| Clase    | Peso   | Interpretación |
|----------|--------|----------------|
| disgust  | 9.4066 | Muy subrepresentada — el modelo pagará 9x más por equivocarse |
| surprise | 1.2934 | Ligeramente subrepresentada |
| angry    | 1.0266 | Balanceada |
| fear     | 1.0010 | Balanceada |
| neutral  | 0.8260 | Ligeramente sobrerrepresentada |
| sad      | 0.8491 | Ligeramente sobrerrepresentada |
| happy    | 0.5684 | Más abundante — menor penalización |

**Referencia:** peso = 1.0 es neutro.
- Mayor a 1.0 → clase subrepresentada → mayor penalización en el loss
- Menor a 1.0 → clase sobrerrepresentada → menor penalización en el loss

Estos pesos se pasan directamente al `CrossEntropyLoss` para que el modelo
no ignore las clases escasas durante el entrenamiento.



## Configuración del Entrenamiento

### Estrategia: Early Stopping + Checkpoint

En lugar de un número fijo de épocas, usamos **Early Stopping** para
detener el entrenamiento automáticamente cuando el modelo deje de mejorar.

**Checkpoint:** guarda el modelo cada vez que `val_accuracy` mejora.
Así siempre conservamos la mejor versión, aunque el entrenamiento continúe.

**Early Stopping:** se basa en `val_loss`. Si después de 5 épocas consecutivas
el val_loss no mejora en al menos `1e-3`, el entrenamiento se detiene.
Esto evita overfitting sin depender del accuracy que puede fluctuar.
```
Época 1  → val_loss mejora → contador = 0
Época 2  → val_loss mejora → contador = 0
Época 3  → val_loss no mejora suficiente | contador = 1
...
Época N  → val_loss no mejora suficiente | contador = 5 → STOP
```

### Hiperparámetros

| Parámetro | Valor | Justificación |
|---|---|---|
| Épocas máximas | 40 | Early stopping detendrá antes si converge |
| Patience | 5 | Épocas sin mejora en val_loss antes de detener |
| Learning rate | 1e-5 | Lion requiere LR más pequeño que AdamW |
| Weight decay | 1e-2 | Recomendado por los autores de Lion |
| Optimizer | Lion | Más eficiente que AdamW para fine-tuning |
| Scheduler | ReduceLROnPlateau | Reduce LR cuando val_loss se estanca |
| MIN_DELTA_LOSS | 1e-3 | Mejora mínima requerida para resetear patience |

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from lion_pytorch import Lion

# Configuración
NUM_EPOCHS   = 40
PATIENCE     = 5
LR           = 1e-5
WEIGHT_DECAY = 1e-2
ckpt_path    = '/content/drive/MyDrive/Deep_learning/proyecto_final/mejor_modelo.pth'

# Variables de control para early stopping
mejor_acc         = 0.0
mejor_val_loss    = float('inf')
epocas_sin_mejora = 0
MIN_DELTA_LOSS = 1e-3

# Loss, optimizer, scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = Lion(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2,threshold=1e-3)

print(f'Épocas máximas:  {NUM_EPOCHS}')
print(f'Early stopping:  {PATIENCE} épocas sin mejora en val_loss')
print(f'Checkpoint en:   {ckpt_path}')
print(f'LR inicial:      {LR}')
print(f'Loss:            CrossEntropyLoss con class weights')
print(f'Optimizer:       Lion')
print(f'Scheduler:       ReduceLROnPlateau (val loss)')

Épocas máximas:  40
Early stopping:  5 épocas sin mejora en val_loss
Checkpoint en:   /content/drive/MyDrive/Deep_learning/proyecto_final/mejor_modelo.pth
LR inicial:      1e-05
Loss:            CrossEntropyLoss con class weights
Optimizer:       Lion
Scheduler:       ReduceLROnPlateau (val loss)


In [ ]:
# Entrenamiento (y progreso)
from tqdm import tqdm

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    pbar = tqdm(loader, desc='Entrenando', leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'acc': f'{100.*correct/total:.2f}%'})

    return running_loss / len(loader), 100. * correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

    return running_loss / len(loader), 100. * correct / total


In [ ]:
# Entrenamiento

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = eval_epoch(model, test_loader, criterion, device)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoca [{epoch+1:03d}/{NUM_EPOCHS}] '
          f'Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%')

    # Checkpoint por val_acc
    if val_acc > mejor_acc:
        mejor_acc = val_acc
        torch.save(model.state_dict(), ckpt_path)
        print(f'  --> Modelo guardado (val_acc: {mejor_acc:.2f}%)')

    # Early stopping por val_loss
    if val_loss < (mejor_val_loss - MIN_DELTA_LOSS):
        mejor_val_loss = val_loss
        epocas_sin_mejora = 0
        print(f'  --> Mejor val_loss: {mejor_val_loss:.4f}')
    else:
        epocas_sin_mejora += 1
        print(f'  --> Sin mejora en val_loss ({epocas_sin_mejora}/{PATIENCE})')

    if epocas_sin_mejora >= PATIENCE:
        print(f'\nEarly stopping en epoca {epoch+1} (val_loss sin mejora)')
        break

print(f'\nEntrenamiento finalizado. Mejor val_acc: {mejor_acc:.2f}% | Mejor val_loss: {mejor_val_loss:.4f}')

Epoca [001/40] Train Loss: 0.0859 Acc: 97.32% | Val Loss: 1.3891 Acc: 67.39%
  --> Modelo guardado (val_acc: 67.39%)
  --> Mejor val_loss: 1.3891


Epoca [002/40] Train Loss: 0.0785 Acc: 97.60% | Val Loss: 1.4415 Acc: 67.80%
  --> Modelo guardado (val_acc: 67.80%)
  --> Sin mejora en val_loss (1/5)


Epoca [003/40] Train Loss: 0.0722 Acc: 97.66% | Val Loss: 1.4727 Acc: 67.23%
  --> Sin mejora en val_loss (2/5)


Epoca [004/40] Train Loss: 0.0705 Acc: 97.86% | Val Loss: 1.4996 Acc: 67.46%
  --> Sin mejora en val_loss (3/5)


Epoca [005/40] Train Loss: 0.0525 Acc: 98.51% | Val Loss: 1.5013 Acc: 67.68%
  --> Sin mejora en val_loss (4/5)


Epoca [006/40] Train Loss: 0.0400 Acc: 98.94% | Val Loss: 1.4854 Acc: 68.28%
  --> Modelo guardado (val_acc: 68.28%)
  --> Sin mejora en val_loss (5/5)

Early stopping en epoca 6 (val_loss sin mejora)

Entrenamiento finalizado. Mejor val_acc: 68.28% | Mejor val_loss: 1.3891
